# 2 · Versions and rules

A partner sends a wrong age. How do we see it, undo it, and stop it next time?

In [ ]:
#@title Setup — run before the session
import os, shutil, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark==3.5.3', 'delta-spark==3.3.2'], check=True)
if not shutil.which('java'):
    subprocess.run('apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless', shell=True, check=True)

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (SparkSession.builder.master('local[2]')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.databricks.delta.properties.defaults.enableDeletionVectors', 'false')
    .config('spark.ui.showConsoleProgress', 'false'))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('OFF')  # errors still reach Python

PATH = '/tmp/delta_talk/versions'
shutil.rmtree(PATH, ignore_errors=True)
print('Spark', spark.version, 'ready')

Spark 3.5.3 ready


In [ ]:
#@title Helpers
def show(version=None):
    reader = spark.read.format('delta')
    if version is not None:
        reader = reader.option('versionAsOf', version)
    reader.load(PATH).orderBy('id').show()

def files_in_table(version=None):
    reader = spark.read.format('delta')
    if version is not None:
        reader = reader.option('versionAsOf', version)
    return sorted(os.path.basename(f) for f in reader.load(PATH).inputFiles())

def reason(error):
    line = next(l for l in str(error).splitlines() if '[DELTA_' in l)
    return line[line.index('[DELTA_'):]

## Same table as before: four users, then user 1 moves to News

In [ ]:
profiles = spark.createDataFrame(
    [(1, 'Male', 25, 'Sports'), (2, 'Female', 30, 'News'),
     (3, 'Male', 35, 'Social'), (4, 'Female', 28, 'Fashion')],
    'id INT, gender STRING, age INT, interest STRING')
profiles.coalesce(1).write.format('delta').save(PATH)

table = DeltaTable.forPath(spark, PATH)
table.update("id = 1", {'interest': "'News'"})
show()

+---+------+---+--------+
| id|gender|age|interest|
+---+------+---+--------+
|  1|  Male| 25|    News|
|  2|Female| 30|    News|
|  3|  Male| 35|  Social|
|  4|Female| 28| Fashion|
+---+------+---+--------+



## A partner sends age 999 for user 2

In [ ]:
table.update("id = 2", {'age': '999'})
show()

+---+------+---+--------+
| id|gender|age|interest|
+---+------+---+--------+
|  1|  Male| 25|    News|
|  2|Female|999|    News|
|  3|  Male| 35|  Social|
|  4|Female| 28| Fashion|
+---+------+---+--------+



## Every commit remembers which version it started from

In [ ]:
table.history().select('version', 'operation', 'readVersion', 'isolationLevel').show()

+-------+---------+-----------+--------------+
|version|operation|readVersion|isolationLevel|
+-------+---------+-----------+--------------+
|      2|   UPDATE|          1|  Serializable|
|      1|   UPDATE|          0|  Serializable|
|      0|    WRITE|       NULL|  Serializable|
+-------+---------+-----------+--------------+



## What did the table look like before the bad batch?

In [ ]:
show(version=1)

+---+------+---+--------+
| id|gender|age|interest|
+---+------+---+--------+
|  1|  Male| 25|    News|
|  2|Female| 30|    News|
|  3|  Male| 35|  Social|
|  4|Female| 28| Fashion|
+---+------+---+--------+



In [ ]:
spark.sql(f"""
  SELECT id, before.age AS age_v1, now.age AS age_now
  FROM delta.`{PATH}` VERSION AS OF 1 AS before
  JOIN delta.`{PATH}` AS now USING (id)
""").show()

+---+------+-------+
| id|age_v1|age_now|
+---+------+-------+
|  1|    25|     25|
|  2|    30|    999|
|  3|    35|     35|
|  4|    28|     28|
+---+------+-------+



## Undo: restore to version 1

In [ ]:
table.restoreToVersion(1)
show()

+---+------+---+--------+
| id|gender|age|interest|
+---+------+---+--------+
|  1|  Male| 25|    News|
|  2|Female| 30|    News|
|  3|  Male| 35|  Social|
|  4|Female| 28| Fashion|
+---+------+---+--------+



## Did restore copy any data?

In [ ]:
print('table now      :', files_in_table())
print('table at v1    :', files_in_table(version=1))
print('table at v2    :', files_in_table(version=2))

table now      : ['part-00000-005fdb9e-c6d2-4d8b-80ec-c78e7dad5f7e-c000.snappy.parquet']
table at v1    : ['part-00000-005fdb9e-c6d2-4d8b-80ec-c78e7dad5f7e-c000.snappy.parquet']
table at v2    : ['part-00000-6bbca817-34d8-4c39-984c-422ca4c05fe9-c000.snappy.parquet']


In [ ]:
table.history().select('version', 'operation', 'operationParameters').show(truncate=False)

+-------+---------+------------------------------------------+
|version|operation|operationParameters                       |
+-------+---------+------------------------------------------+
|3      |RESTORE  |{version -> 1, timestamp -> NULL}         |
|2      |UPDATE   |{predicate -> ["(id#224 = 2)"]}           |
|1      |UPDATE   |{predicate -> ["(id#224 = 1)"]}           |
|0      |WRITE    |{mode -> ErrorIfExists, partitionBy -> []}|
+-------+---------+------------------------------------------+



## Stop it happening again: put the rule on the table

In [ ]:
spark.sql(f"ALTER TABLE delta.`{PATH}` ADD CONSTRAINT valid_age CHECK (age BETWEEN 0 AND 120)")

DataFrame[]

In [ ]:
try:
    table.update("id = 2", {'age': '999'})
except Exception as e:
    print(reason(e))

[DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint valid_age ((age >= 0) AND (age <= 120)) violated by row with values:


In [ ]:
table.history().select('version', 'operation').show()
show()

+-------+--------------+
|version|     operation|
+-------+--------------+
|      4|ADD CONSTRAINT|
|      3|       RESTORE|
|      2|        UPDATE|
|      1|        UPDATE|
|      0|         WRITE|
+-------+--------------+

+---+------+---+--------+
| id|gender|age|interest|
+---+------+---+--------+
|  1|  Male| 25|    News|
|  2|Female| 30|    News|
|  3|  Male| 35|  Social|
|  4|Female| 28| Fashion|
+---+------+---+--------+



**The table now has versions and rules. What happens when it gets big — do we read every file?** → next